# Notebook 1.4: Displaced Squeezed States and Phase Space Interference
### Module 1: Single-Mode Quantum States of Light
**Computational Quantum Optics Curriculum**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rfandan/computational-quantum-optics/blob/main/module_1_states_of_light/05_displaced_squeezed_states.ipynb)

---

### Prerequisites
- Notebook 1.1: Fock States, Ladder Operators, and Truncation Artifacts
- Notebook 1.2: Coherent States, Displacement, and Poissonian Statistics
- Notebook 1.3: Squeezed Vacuum and Quadrature Noise Reduction

### Pedagogical Learning Objectives
1. **Mathematical Definition & Operator Ordering:** Construct displaced squeezed states $|\alpha, \xi\rangle = \hat{D}(\alpha)\hat{S}(\xi)|0\rangle$ and analyze the operator algebra relating $\hat{D}(\alpha)\hat{S}(\xi)$ to $\hat{S}(\xi)\hat{D}(\alpha)$.
2. **Quadrature Fluctuations:** Calculate expectation values $\langle \hat{X}_1 \rangle, \langle \hat{X}_2 \rangle$ and variances $\langle (\Delta \hat{X}_1)^2 \rangle, \langle (\Delta \hat{X}_2)^2 \rangle$, illustrating the displacement of the uncertainty ellipse in phase space.
3. **Amplitude vs. Phase Squeezing:** Contrast radial (amplitude) noise suppression ($Q < 0$, sub-Poissonian) with tangential (phase) noise suppression ($Q > 0$, super-Poissonian).
4. **Schleich-Wheeler Phase Space Interference:** Uncover the physical origin of the rapid oscillations in the photon number distribution $P_n$ via the area-of-overlap principle between the displaced ellipse and the Fock state rings.
5. **Numerical Implementation & QuTiP Verification:** Construct robust solvers in truncated Fock space using NumPy/SciPy matrix exponentials and benchmark against QuTiP.


In [ ]:
# Setup, environment detection, and library imports
import sys
import math
import numpy as np
import scipy.linalg as la
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

# Plot styling configuration for high-DPI publication graphics
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 10
plt.rcParams['lines.linewidth'] = 2.0
plt.rcParams['grid.alpha'] = 0.35

# Detect QuTiP availability with clean fallback
try:
    import qutip as qt
    HAS_QUTIP = True
    print(f"QuTiP {qt.__version__} successfully imported.")
except ImportError:
    HAS_QUTIP = False
    print("QuTiP not found. Using native NumPy/SciPy implementations.")

# Detect interactive widgets environment
try:
    from ipywidgets import interact, FloatSlider, IntSlider
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False
    print("ipywidgets not found. Interactive exploration will run in static fallback mode.")


## 1. Mathematical Architecture of Displaced Squeezed States

A **displaced squeezed state** $|\alpha, \xi\rangle$ is synthesized by first squeezing the vacuum state $|0\rangle$ and subsequently applying the unitary displacement operator $\hat{D}(\alpha)$:

$$
|\alpha, \xi\rangle = \hat{D}(\alpha) \hat{S}(\xi) |0\rangle
$$

where:
- $\hat{D}(\alpha) = \exp\left(\alpha \hat{a}^\dagger - \alpha^* \hat{a}\right)$ with complex displacement amplitude $\alpha = |\alpha| e^{i\phi}$.
- $\hat{S}(\xi) = \exp\left[\frac{1}{2}\left(\xi^* \hat{a}^2 - \xi (\hat{a}^\dagger)^2\right)\right]$ with complex squeezing parameter $\xi = r e^{i\theta}$.

---

### Operator Commutation & Displaced Squeezed vs. Squeezed Coherent States

Does the sequence of operations matter? Consider applying displacement first, followed by squeezing:
$$
|\psi_{\text{SC}}\rangle = \hat{S}(\xi) \hat{D}(\alpha) |0\rangle
$$
Using the Bogoliubov transformation generated by $\hat{S}(\xi)$:
$$
\hat{S}^\dagger(\xi) \hat{a} \hat{S}(\xi) = \hat{a} \cosh r - \hat{a}^\dagger e^{i\theta} \sinh r
$$
we insert resolution of identity $\hat{S}\hat{S}^\dagger = \hat{I}$:
$$
\hat{S}(\xi) \hat{D}(\alpha) \hat{S}^\dagger(\xi) = \hat{D}(\gamma) \quad \Longrightarrow \quad \hat{S}(\xi) \hat{D}(\alpha) = \hat{D}(\gamma) \hat{S}(\xi)
$$
where the transformed displacement amplitude $\gamma$ is:
$$
\gamma = \alpha \cosh r - \alpha^* e^{i\theta} \sinh r
$$

Therefore, a **squeezed coherent state** $\hat{S}(\xi)\hat{D}(\alpha)|0\rangle$ is identically a **displaced squeezed state** $\hat{D}(\gamma)\hat{S}(\xi)|0\rangle$, but with an amplified or demagnified displacement parameter $\gamma$! In the literature, the standard convention $|\alpha, \xi\rangle = \hat{D}(\alpha)\hat{S}(\xi)|0\rangle$ is preferred because $\alpha$ represents the observable phase-space center: $\langle \hat{a} \rangle = \alpha$.

---

### Quadrature Moments and Uncertainty Ellipse

Recalling the dimensionless quadrature operators $\hat{X}_1 = \frac{\hat{a} + \hat{a}^\dagger}{2}$ and $\hat{X}_2 = \frac{\hat{a} - \hat{a}^\dagger}{2i}$:

1. **Mean Values:**
   $$
   \langle \hat{X}_1 \rangle = \text{Re}(\alpha), \qquad \langle \hat{X}_2 \rangle = \text{Im}(\alpha)
   $$
   The displacement operator strictly translates the center of the quantum state in phase space without altering its shape or quantum noise.

2. **Variances:**
   For squeezing angle $\theta = 0$ (squeezing along $\hat{X}_1$):
   $$
   \langle (\Delta \hat{X}_1)^2 \rangle = \frac{1}{4} e^{-2r}, \qquad \langle (\Delta \hat{X}_2)^2 \rangle = \frac{1}{4} e^{2r}
   $$
   The uncertainty product remains minimally saturated:
   $$
   \Delta \hat{X}_1 \Delta \hat{X}_2 = \frac{1}{4}
   $$
   A displaced squeezed state remains a **minimum-uncertainty state**!


In [ ]:
# Numerical Construction of Displaced Squeezed States in Truncated Fock Space

def annihilation_operator(N):
    """Constructs truncated Fock space annihilation operator a."""
    diag_vals = np.sqrt(np.arange(1, N, dtype=np.float64))
    return np.diag(diag_vals, k=1)

def displacement_operator(alpha, N):
    """Constructs unitary displacement operator D(alpha) = exp(alpha a^dagger - alpha* a)."""
    a = annihilation_operator(N)
    adag = a.T.conj()
    generator = alpha * adag - np.conj(alpha) * a
    return la.expm(generator)

def squeeze_operator(xi, N):
    """Constructs unitary squeeze operator S(xi) = exp(0.5 * (xi* a^2 - xi (a^dagger)^2))."""
    a = annihilation_operator(N)
    adag = a.T.conj()
    a2 = a @ a
    adag2 = adag @ adag
    generator = 0.5 * (np.conj(xi) * a2 - xi * adag2)
    return la.expm(generator)

def displaced_squeezed_state(alpha, r, theta, N):
    """Generates displaced squeezed state |alpha, xi> = D(alpha) S(xi) |0>."""
    xi = r * np.exp(1j * theta)
    vac = np.zeros(N, dtype=np.complex128)
    vac[0] = 1.0  # Vacuum state |0>
    
    # Unitary operations
    S = squeeze_operator(xi, N)
    D = displacement_operator(alpha, N)
    
    # Sequential application: D * S * |0>
    psi_sq = S @ vac
    psi = D @ psi_sq
    
    norm = np.linalg.norm(psi)
    norm_loss = abs(1.0 - norm)
    tail_prob = abs(psi[-1])**2
    
    return psi, norm_loss, tail_prob

# Verification of operator ordering identity: S(xi) D(alpha) |0> == D(gamma) S(xi) |0>
N_test = 50
alpha_test = 2.0 + 1.0j
r_test = 0.6
theta_test = np.pi / 4.0
xi_test = r_test * np.exp(1j * theta_test)

# S * D * |0>
S_mat = squeeze_operator(xi_test, N_test)
D_mat = displacement_operator(alpha_test, N_test)
vac_test = np.zeros(N_test, dtype=np.complex128)
vac_test[0] = 1.0

psi_SD = S_mat @ (D_mat @ vac_test)

# Analytical gamma = alpha * cosh(r) - alpha* * exp(i*theta) * sinh(r)
gamma = alpha_test * np.cosh(r_test) - np.conj(alpha_test) * np.exp(1j * theta_test) * np.sinh(r_test)
D_gamma = displacement_operator(gamma, N_test)
psi_DS = D_gamma @ (S_mat @ vac_test)

# Quantum state fidelity F = |<psi_SD | psi_DS>|^2
overlap = np.vdot(psi_SD, psi_DS)
fidelity = abs(overlap)**2

print(f"--- Operator Ordering Verification ---")
print(f"Displacement alpha: {alpha_test}")
print(f"Transformed gamma:  {gamma.real:.4f} + {gamma.imag:.4f}j")
print(f"State overlap |<S D 0 | D(gamma) S 0>|^2: {fidelity:.8f}")
assert abs(fidelity - 1.0) < 1e-4, "Operator commutation identity verification failed!"
print("Identity S(xi)D(alpha) = D(gamma)S(xi) mathematically confirmed!")


## 2. Amplitude Squeezing vs. Phase Squeezing

The orientation of the squeezed noise ellipse relative to the displacement vector $\alpha$ fundamentally dictates the photon counting statistics of the field:

1. **Mean Photon Number:**
   $$
   \langle \hat{n} \rangle = \langle \hat{a}^\dagger \hat{a} \rangle = |\alpha|^2 + \sinh^2 r
   $$
   The total energy decomposes into the coherent field intensity $|\alpha|^2$ plus the parametric photon excitation $\sinh^2 r$.

2. **Photon Number Variance:**
   $$
   \langle (\Delta \hat{n})^2 \rangle = |\alpha|^2 \left[ \cosh 2r - \sinh 2r \cos(2\phi - \theta) \right] + 2\sinh^2 r \cosh^2 r
   $$
   where $\phi = \arg(\alpha)$ is the displacement angle and $\theta = \arg(\xi)$ is the squeeze direction.

---

### Physical Regime Classification

- **Amplitude (Radial) Squeezing ($2\phi - \theta = 0$):**
  When the squeezed axis aligns with the radial displacement direction:
  $$
  \langle (\Delta \hat{n})^2 \rangle = |\alpha|^2 e^{-2r} + 2\sinh^2 r \cosh^2 r
  $$
  For significant coherent amplitude ($|\alpha| \gg \sinh r$), the photon number variance drops well below the Poissonian shot noise limit:
  $$
  \langle (\Delta \hat{n})^2 \rangle < \langle \hat{n} \rangle \quad \Longleftrightarrow \quad \text{Sub-Poissonian Light} \ (Q < 0)
  $$
  The intensity fluctuations of the laser field are actively suppressed below the standard quantum limit!

- **Phase (Tangential) Squeezing ($2\phi - \theta = \pi$):**
  When the noise ellipse is oriented perpendicular to the displacement vector, phase fluctuations are suppressed while radial (intensity) fluctuations are stretched:
  $$
  \langle (\Delta \hat{n})^2 \rangle = |\alpha|^2 e^{2r} + 2\sinh^2 r \cosh^2 r \gg \langle \hat{n} \rangle \quad \Longleftrightarrow \quad \text{Super-Poissonian Light} \ (Q > 0)
  $$
  Phase squeezing enables ultra-precise optical interferometry (e.g. gravitational wave detection in LIGO) by minimizing optical phase jitter at the expense of increased photon number uncertainty.


In [ ]:
# Comparative Simulation: Amplitude Squeezing vs. Phase Squeezing

N_dim = 70
alpha_val = 3.0  # Real displacement along X1 (phi = 0)
r_val = 0.55     # ~4.8 dB squeezing

# Case A: Amplitude Squeezing (theta = 0)
# Noise ellipse squeezed along X1 (radial direction)
psi_amp, err_amp, _ = displaced_squeezed_state(alpha_val, r_val, theta=0.0, N=N_dim)
p_n_amp = np.abs(psi_amp)**2

# Case B: Phase Squeezing (theta = np.pi)
# Noise ellipse squeezed along X2 (transverse direction, stretched along X1)
psi_phase, err_phase, _ = displaced_squeezed_state(alpha_val, r_val, theta=np.pi, N=N_dim)
p_n_phase = np.abs(psi_phase)**2

# Reference: Coherent State |alpha>
psi_coh, _, _ = displaced_squeezed_state(alpha_val, r=0.0, theta=0.0, N=N_dim)
p_n_coh = np.abs(psi_coh)**2

n_indices = np.arange(N_dim)

# Moments calculation
n_op = np.diag(n_indices)
mean_n_amp = np.real(np.vdot(psi_amp, n_op @ psi_amp))
var_n_amp = np.real(np.vdot(psi_amp, (n_op @ n_op) @ psi_amp)) - mean_n_amp**2
q_amp = (var_n_amp - mean_n_amp) / mean_n_amp

mean_n_phase = np.real(np.vdot(psi_phase, n_op @ psi_phase))
var_n_phase = np.real(np.vdot(psi_phase, (n_op @ n_op) @ psi_phase)) - mean_n_phase**2
q_phase = (var_n_phase - mean_n_phase) / mean_n_phase

print(f"Amplitude Squeezing: <n> = {mean_n_amp:.3f}, Var(n) = {var_n_amp:.3f}, Mandel Q = {q_amp:.3f} (Sub-Poissonian)")
print(f"Phase Squeezing:     <n> = {mean_n_phase:.3f}, Var(n) = {var_n_phase:.3f}, Mandel Q = {q_phase:.3f} (Super-Poissonian)")

# Plotting: Two-panel figure comparing phase space ellipses and photon distributions
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

# Panel 1: Phase Space Noise Ellipses (1-sigma contour)
circle_coh = plt.Circle((alpha_val, 0), 0.5, color='#2ca02c', fill=False, lw=2, ls='--', label=r'Coherent ($r=0$)')
ellipse_amp = Ellipse((alpha_val, 0), width=np.exp(-r_val), height=np.exp(r_val), angle=0,
                      edgecolor='#1f77b4', facecolor='#1f77b4', alpha=0.35, lw=2.5,
                      label=r'Amplitude Squeezed ($\theta=0$)')
ellipse_phase = Ellipse((alpha_val, 0), width=np.exp(r_val), height=np.exp(-r_val), angle=0,
                        edgecolor='#d62728', facecolor='#d62728', alpha=0.25, lw=2.5,
                        label=r'Phase Squeezed ($\theta=\pi$)')

ax1.add_patch(ellipse_amp)
ax1.add_patch(ellipse_phase)
ax1.add_patch(circle_coh)
ax1.plot(alpha_val, 0, 'ko', markersize=6, label=r'Mean $\langle \hat{a} \rangle = 3.0$')

# Radial vector arrow
ax1.annotate('', xy=(alpha_val, 0), xytext=(0, 0),
             arrowprops=dict(arrowstyle="->", color="black", lw=1.5, ls=':'))

ax1.set_xlim(-1, 5.5)
ax1.set_ylim(-2.5, 2.5)
ax1.set_aspect('equal')
ax1.set_xlabel(r"Quadrature $\hat{X}_1$")
ax1.set_ylabel(r"Quadrature $\hat{X}_2$")
ax1.set_title(r"Phase Space Quadrature Noise Ellipses ($1\sigma$ Contours)")
ax1.legend(loc='upper left', framealpha=0.9)
ax1.grid(True)

# Panel 2: Photon Number Distributions P(n)
n_max_plot = 35
ax2.plot(n_indices[:n_max_plot], p_n_coh[:n_max_plot], 'go--', lw=1.5, markersize=5,
         label=rf'Coherent: Var$(n)={alpha_val**2:.1f}, Q=0$')
ax2.plot(n_indices[:n_max_plot], p_n_amp[:n_max_plot], 's-', color='#1f77b4', lw=2.0, markersize=6,
         label=rf'Amplitude Squeezed: Var$(n)={var_n_amp:.2f}, Q={q_amp:.2f}$')
ax2.plot(n_indices[:n_max_plot], p_n_phase[:n_max_plot], '^-.', color='#d62728', lw=2.0, markersize=6,
         label=rf'Phase Squeezed: Var$(n)={var_n_phase:.2f}, Q=+{q_phase:.2f}$')

ax2.set_xlim(-0.5, n_max_plot)
ax2.set_xlabel(r"Photon Number $n$")
ax2.set_ylabel(r"Probability $P(n) = |\langle n | \psi \rangle|^2$")
ax2.set_title("Photon Counting Statistics: Sub- vs. Super-Poissonian Light")
ax2.legend(loc='upper right', framealpha=0.9)
ax2.grid(True)

plt.tight_layout()
plt.show()


## 3. Schleich-Wheeler Phase Space Interference

Why does the photon number distribution of a displaced squeezed state exhibit striking, rapid oscillations?

In 1987–1988, Wolfgang Schleich and John Archibald Wheeler published a seminal discovery (*Nature* 334, 141–147, 1988): **quantum oscillations in the photon number distribution are a direct manifestation of interference in phase space!**

---

### The Area-of-Overlap Principle

In quantum optics, the transition probability between two states $|\psi\rangle$ and $|\phi\rangle$ can be visualized through the geometric overlap of their phase-space representations:

1. **Geometry of Fock State $|n\rangle$:**
   In phase space, a number state $|n\rangle$ is an azimuthally symmetric ring of radius $r_n = \sqrt{n + 1/2}$ and annular thickness $\delta r \approx 1/(2\sqrt{n})$.
2. **Geometry of Displaced Squeezed State $|\alpha, \xi\rangle$:**
   The state is an elongated Gaussian uncertainty ellipse centered at $\alpha$.
3. **Double Intersection:**
   For a range of photon numbers $n$, the narrow ellipse intersects the circular ring of $|n\rangle$ at **two distinct regions** in phase space:
   $$
   \mathcal{A}_1 \quad \text{and} \quad \mathcal{A}_2
   $$
4. **Quantum Interference:**
   Because probability amplitudes add coherently ($\langle n | \psi \rangle = \mathcal{A}_1 e^{i\theta_1} + \mathcal{A}_2 e^{i\theta_2}$), the total probability $P_n$ is:
   $$
   P_n = |\langle n | \alpha, \xi \rangle|^2 = |\mathcal{A}_1|^2 + |\mathcal{A}_2|^2 + 2|\mathcal{A}_1||\mathcal{A}_2| \cos(\Delta \Phi_n)
   $$
   where $\Delta \Phi_n = \theta_1 - \theta_2$ is the geometric phase area enclosed between the ellipse and the circle in phase space!

As $n$ increments by 1, the radius $\sqrt{n+1/2}$ expands, causing $\Delta \Phi_n$ to alternate between constructive ($\cos \Delta \Phi = +1$) and destructive ($\cos \Delta \Phi = -1$) interference. This produces dramatic, non-classical oscillatory fringes in $P_n$ that cannot occur in any classical probability distribution!


In [ ]:
# Simulating Schleich-Wheeler Interference in Phase Space

N_osc = 120
alpha_osc = 5.0    # Coherent displacement
r_osc = 0.85       # Substantial squeezing parameter (~7.4 dB)
theta_osc = 0.0    # Squeezed along X1, stretched along X2

psi_sw, norm_err, tail = displaced_squeezed_state(alpha_osc, r_osc, theta_osc, N=N_osc)
p_n_sw = np.abs(psi_sw)**2
n_sw = np.arange(N_osc)

# Mean and variance
mean_n_sw = np.sum(n_sw * p_n_sw)
var_n_sw = np.sum((n_sw - mean_n_sw)**2 * p_n_sw)

print(f"Norm error: {norm_err:.2e}, Tail probability: {tail:.2e}")
print(f"Mean photon number <n> = {mean_n_sw:.2f}, Photon variance Var(n) = {var_n_sw:.2f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

# Panel 1: Geometric Phase Space Overlap (Area of Overlap concept)
ellipse_sw = Ellipse((alpha_osc, 0), width=2*np.exp(-r_osc), height=2*np.exp(r_osc), angle=0,
                     edgecolor='navy', facecolor='cyan', alpha=0.45, lw=2.5,
                     label=r'State $|\alpha, r\rangle$ ($\alpha=5, r=0.85$)')
ax1.add_patch(ellipse_sw)
ax1.plot(alpha_osc, 0, 'bo', markersize=6)

# Overlay several Fock state rings r = sqrt(n + 0.5)
fock_test_n = [15, 25, 35]
colors_fock = ['#e41a1c', '#377eb8', '#4daf4a']
for n_val, col in zip(fock_test_n, colors_fock):
    radius = np.sqrt(n_val + 0.5)
    ring = plt.Circle((0, 0), radius, color=col, fill=False, lw=1.8, ls='--',
                      label=rf'Fock ring $|{n_val}\rangle$ ($R=\sqrt{{{n_val}+0.5}}\approx{radius:.2f}$)')
    ax1.add_patch(ring)

ax1.set_xlim(-1, 8.5)
ax1.set_ylim(-5.5, 5.5)
ax1.set_aspect('equal')
ax1.set_xlabel(r"Quadrature $\hat{X}_1$")
ax1.set_ylabel(r"Quadrature $\hat{X}_2$")
ax1.set_title("Geometric Origin: Overlap of Ellipse with Fock Rings")
ax1.legend(loc='upper left', framealpha=0.9, fontsize=9)
ax1.grid(True)

# Panel 2: Rapid Oscillations in P(n)
n_view = 65
ax2.plot(n_sw[:n_view], p_n_sw[:n_view], 'o-', color='#003366', lw=2.0, markersize=5,
         label=r'Displaced Squeezed $P(n) = |\langle n | \alpha, \xi \rangle|^2$')
ax2.axvline(mean_n_sw, color='red', ls=':', lw=2, label=rf'Mean $\langle n \rangle = {mean_n_sw:.1f}$')

ax2.set_xlim(-0.5, n_view)
ax2.set_xlabel(r"Photon Number $n$")
ax2.set_ylabel(r"Probability $P(n)$")
ax2.set_title("Schleich-Wheeler Oscillations in Photon Distribution")
ax2.legend(loc='upper right', framealpha=0.9)
ax2.grid(True)

plt.tight_layout()
plt.show()


> **Physical Insight: Classical vs. Quantum Interference in Phase Space**
> In classical statistical mechanics, a phase-space distribution represents an ensemble of particles with positive probability density $W(x, p) \ge 0$. Integrating $W(x, p)$ over any phase-space ring yields an unconditionally smooth, monotonic distribution with zero fringes.
>
> In quantum mechanics, however, **phase space possesses an underlying phase**! The transition amplitude $\langle n | \psi \rangle$ is an integral of complex wavefunctions whose quantum phases interfere. The two intersection lobes in phase space act precisely like the two pinholes in Young's double-slit experiment, producing interference fringes in photon number space rather than physical position space!


In [ ]:
# Interactive Exploration: Phase Space Geometry & Quantum Statistics

def explore_displaced_squeezed(alpha=3.0, r=0.5, theta_deg=0.0):
    """Interactive simulation: renders noise ellipse and photon distribution."""
    theta_rad = np.radians(theta_deg)
    N_calc = 70
    
    psi, err, _ = displaced_squeezed_state(alpha, r, theta_rad, N_calc)
    p_n = np.abs(psi)**2
    n_arr = np.arange(N_calc)
    
    mean_n = np.sum(n_arr * p_n)
    var_n = np.sum((n_arr - mean_n)**2 * p_n)
    q_param = (var_n - mean_n) / mean_n if mean_n > 1e-5 else 0.0
    
    # Quadrature standard deviations (principal axes)
    std_squeezed = 0.5 * np.exp(-r)
    std_stretched = 0.5 * np.exp(r)
    squeeze_db = 10.0 * np.log10(np.exp(2 * r))
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    
    # 1. Phase space ellipse
    ellipse = Ellipse((alpha, 0), width=2*std_squeezed, height=2*std_stretched,
                      angle=np.degrees(theta_rad / 2.0),
                      edgecolor='#d62728', facecolor='#ff7f0e', alpha=0.4, lw=2.5,
                      label=f'Noise Ellipse ({squeeze_db:.1f} dB)')
    ax1.add_patch(ellipse)
    ax1.plot(alpha, 0, 'ro', markersize=6, label=rf'$\langle \hat{{a}} \rangle = {alpha:.1f}$')
    
    # Coherent reference circle
    ref_circ = plt.Circle((alpha, 0), 0.5, color='gray', fill=False, ls='--', lw=1.5, label='Shot Noise Limit')
    ax1.add_patch(ref_circ)
    
    ax1.set_xlim(-1, 6)
    ax1.set_ylim(-3, 3)
    ax1.set_aspect('equal')
    ax1.set_xlabel(r"$\hat{X}_1$")
    ax1.set_ylabel(r"$\hat{X}_2$")
    ax1.set_title(rf"Phase Space Ellipse: $\alpha={alpha:.1f}, r={r:.2f}, \theta={theta_deg:.0f}^\circ$")
    ax1.legend(loc='upper left', fontsize=9)
    ax1.grid(True)
    
    # 2. Photon number distribution
    n_plot = 40
    bars = ax2.bar(n_arr[:n_plot], p_n[:n_plot], color='#1f77b4', alpha=0.75, edgecolor='black', width=0.8)
    ax2.axvline(mean_n, color='red', ls='--', lw=2, label=rf'$\langle n \rangle = {mean_n:.2f}$')
    
    # Regimes annotation
    stat_type = "Sub-Poissonian (Non-classical)" if q_param < -0.05 else ("Super-Poissonian" if q_param > 0.05 else "Poissonian")
    ax2.set_title(f"Photon Distribution ({stat_type}): Mandel Q = {q_param:+.3f}")
    ax2.set_xlabel("Photon Number n")
    ax2.set_ylabel("P(n)")
    ax2.set_xlim(-0.5, n_plot)
    ax2.legend(loc='upper right')
    ax2.grid(True)
    
    plt.tight_layout()
    plt.show()

# Launch interactive exploration
if HAS_WIDGETS:
    interact(explore_displaced_squeezed,
             alpha=FloatSlider(min=0.5, max=5.0, step=0.5, value=3.0, description=r'Displacement \alpha:'),
             r=FloatSlider(min=0.0, max=1.2, step=0.1, value=0.6, description=r'Squeezing r:'),
             theta_deg=FloatSlider(min=0.0, max=360.0, step=30.0, value=0.0, description=r'Angle \theta (deg):'))
else:
    # Static execution fallback for headless environments
    explore_displaced_squeezed(alpha=3.0, r=0.6, theta_deg=0.0)


## 4. Self-Check Exercises and Pedagogical Challenges

Test your understanding of displaced squeezed states and phase space interference by completing these analytical and computational challenges.

---

### Challenge 1: The Transition from Sub-Poissonian to Super-Poissonian
For a real displacement $\alpha > 0$ and real squeeze parameter $\xi = r > 0$, the photon number variance is:
$$
\langle (\Delta \hat{n})^2 \rangle = \alpha^2 \left( \cosh 2r - \sinh 2r \cos\theta \right) + 2\sinh^2 r \cosh^2 r
$$
1. Prove that for $\theta = 0$ (amplitude squeezing), the state is always sub-Poissonian ($Q < 0$) provided that:
   $$
   \alpha^2 > \frac{\sinh^2 r (2\cosh^2 r - 1)}{1 - e^{-2r}}
   $$
2. What happens to the Mandel $Q$ parameter in the deep squeezing limit $r \to \infty$ for fixed $\alpha$?

<details>
<summary><b>Click to expand Solution to Challenge 1</b></summary>

**Part 1 Solution:**
The condition for sub-Poissonian statistics is $Q < 0$, which is equivalent to $\langle (\Delta \hat{n})^2 \rangle < \langle \hat{n} \rangle$.
The mean photon number is $\langle \hat{n} \rangle = \alpha^2 + \sinh^2 r$.
For $\theta = 0$, $\cosh 2r - \sinh 2r = e^{-2r}$. Thus:
$$
\alpha^2 e^{-2r} + 2\sinh^2 r \cosh^2 r < \alpha^2 + \sinh^2 r
$$
Rearranging terms:
$$
\alpha^2 (1 - e^{-2r}) > 2\sinh^2 r \cosh^2 r - \sinh^2 r = \sinh^2 r (2\cosh^2 r - 1) = \sinh^2 r \cosh 2r
$$
Since $1 - e^{-2r} > 0$ for $r > 0$, dividing yields:
$$
\alpha^2 > \frac{\sinh^2 r \cosh 2r}{1 - e^{-2r}}
$$
This demonstrates that sub-Poissonian noise requires a sufficiently large coherent displacement $\alpha$ to dominate over the intrinsic thermal-like photon fluctuations $2\sinh^2 r \cosh^2 r$ of the squeezed vacuum.

**Part 2 Solution:**
As $r \to \infty$, the squeezed vacuum term $2\sinh^2 r \cosh^2 r \sim \frac{1}{2} e^{4r} / 4$ grows much faster than $\alpha^2 e^{-2r} \to 0$ and $\langle \hat{n} \rangle \sim \frac{1}{4} e^{2r}$.
Consequently, $\langle (\Delta n)^2 \rangle \sim \frac{1}{8} e^{4r} = 2 \langle n \rangle^2$, which yields Mandel $Q \to \infty$ (strongly super-Poissonian). Extreme squeezing always converts the state into a super-Poissonian state due to parametric pair generation!
</details>

---

### Challenge 2: Numerical Verification with QuTiP
Implement a short script that generates $|\alpha, \xi\rangle$ using QuTiP's built-in `displace()` and `squeeze()` functions, and verify that the calculated fidelity $|\langle \psi_{\text{num}} | \psi_{\text{qutip}} \rangle|^2$ between our NumPy implementation and QuTiP is $> 0.9999$.

<details>
<summary><b>Click to expand Solution to Challenge 2</b></summary>

```python
import numpy as np
if HAS_QUTIP:
    import qutip as qt
    N = 60
    alpha = 2.5
    r = 0.5
    theta = np.pi / 3.0
    
    # QuTiP state synthesis
    vac_qt = qt.fock(N, 0)
    psi_qt = qt.displace(N, alpha) * qt.squeeze(N, r * np.exp(1j * theta)) * vac_qt
    
    # NumPy state synthesis
    psi_np, _, _ = displaced_squeezed_state(alpha, r, theta, N)
    
    # Compute fidelity
    fid = qt.fidelity(psi_qt, qt.Qobj(psi_np.reshape(-1, 1)))**2
    print(f"QuTiP vs NumPy Fidelity: {fid:.8f}")
    assert fid > 0.9999, "Fidelity benchmark failed!"
else:
    print("QuTiP not installed; run in Google Colab to execute this test.")
```
</details>


## 5. Summary, Reference Table, and Outlook

### Comprehensive Summary of Displaced Squeezed States

| Property | Squeezed Vacuum $|0, \xi\rangle$ | Coherent State $|\alpha\rangle$ | Displaced Squeezed State $|\alpha, \xi\rangle$ |
| :--- | :--- | :--- | :--- |
| **Phase Space Center** | $(0, 0)$ | $(\text{Re}\,\alpha, \text{Im}\,\alpha)$ | $(\text{Re}\,\alpha, \text{Im}\,\alpha)$ |
| **Noise Geometry** | Squeezed Ellipse | Symmetric Circle ($r_0 = 1/2$) | Squeezed Ellipse |
| **Uncertainty Product** | $\Delta X_1 \Delta X_2 = 1/4$ | $\Delta X_1 \Delta X_2 = 1/4$ | $\Delta X_1 \Delta X_2 = 1/4$ |
| **Mean Photon Number** | $\sinh^2 r$ | $|\alpha|^2$ | $|\alpha|^2 + \sinh^2 r$ |
| **Photon Statistics** | Even-Fock spikes only | Poissonian ($Q = 0$) | Oscillatory (Schleich-Wheeler fringes) |
| **Sub-Poissonian?** | Never ($Q > 0$) | No ($Q = 0$) | Yes ($Q < 0$) for amplitude squeezing |

---

### Key Takeaways
1. **Operator Ordering:** Squeezing then displacing vs. displacing then squeezing are related by the canonical identity $\hat{S}(\xi)\hat{D}(\alpha) = \hat{D}(\gamma)\hat{S}(\xi)$, where $\gamma = \alpha \cosh r - \alpha^* e^{i\theta} \sinh r$.
2. **Phase vs. Amplitude Squeezing:** Aligning the squeeze direction with displacement creates sub-Poissonian light with suppressed intensity noise, while orthogonal alignment yields phase-squeezed light essential for precision interferometry.
3. **Phase Space Interference:** Non-classical oscillations in $P_n$ emerge from the quantum geometric overlap of the elongated noise ellipse with circular Fock state manifolds.

---

### Primary References
- **Schleich, W. P., & Wheeler, J. A.** (1987). *Oscillations in photon distribution: the concept of 'interference in phase space'*. Nature, 334(6178), 141–147.
- **Gerry, C. C., & Knight, P. L.** (2005). *Introductory Quantum Optics*. Cambridge University Press. Chapter 7: Squeezed States of Light.
- **Scully, M. O., & Zubairy, M. S.** (1997). *Quantum Optics*. Cambridge University Press. Chapter 2: Field Quantization and Coherent States.
- **Walls, D. F., & Milburn, G. J.** (2008). *Quantum Optics*. Springer. Chapter 5: Squeezed States.

---

### Next in the Curriculum
In **Notebook 1.5: Thermal States and Mixed Radiation Fields**, we transition from pure quantum states to statistical ensembles, analyzing density matrices $\rho_{\text{th}}$, Bose-Einstein statistics, purity $\mathcal{P} = \mathrm{Tr}(\rho^2) = 1/(2\bar{n}+1)$, and the thermalization of coherent states.
